[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/02_Nodes_Edges_and_Tensors/Nodes_Edges_and_Tensors_Apply.ipynb)

# 3.2 Nodes, Edges, and Tensors — Hands-On Practice

## Objective

Understand the three fundamental building blocks of ONNX graphs: **nodes** (operators),
**edges** (data flow via named tensors), and **tensors** (typed multi-dimensional arrays).

---

| # | Section | Focus |
|---|---------|-------|
| 1 | Setup | Dependencies |
| 2 | Exercise 1: Conv-like Pipeline | Build and inspect a CNN block |
| 3 | Exercise 2: Data Flow Tracing | Track tensor names through the graph |
| 4 | Exercise 3: Tensor Type System | Explore ONNX data types |
| 5 | Exercise 4: NodeProto Inspection | Dissect node structure |
| 6 | Exercise 5: Edge Analysis | Map producer/consumer relationships |
| 7 | Exercise 6: Tensor Shape Manipulation | Reshape, Squeeze, Unsqueeze |
| 8 | Exercise 7: Multi-Output Nodes | Operators producing multiple tensors |
| 9 | Challenge: Full CNN Feature Extractor | Conv → Pool → Flatten → Dense |

In [ ]:
# !pip install onnx onnxruntime numpy matplotlib

import numpy as np
import onnx
from onnx import TensorProto, shape_inference
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid,
    tensor_dtype_to_np_dtype)
from onnx.checker import check_model
from onnx.numpy_helper import from_array, to_array
from onnx.mapping import TENSOR_TYPE_MAP
import onnxruntime as ort

print(f'ONNX: {onnx.__version__}  ORT: {ort.__version__}')

## Exercise 1: Build a Conv-Like Pipeline

Build a small CNN block: `Conv → BatchNorm → Relu → MaxPool`

$$\text{Conv}(X, W) = \sum_{c,k_h,k_w} W_{c,k_h,k_w} \cdot X_{c, h+k_h, w+k_w}$$

In [ ]:
np.random.seed(42)
IN_C, OUT_C = 3, 16

# Initializers
conv_W = from_array(
    np.random.randn(OUT_C, IN_C, 3, 3).astype(np.float32) * 0.1, 'conv_W')
conv_B = from_array(np.zeros(OUT_C, dtype=np.float32), 'conv_B')
bn_scale = from_array(np.ones(OUT_C, dtype=np.float32), 'bn_scale')
bn_bias = from_array(np.zeros(OUT_C, dtype=np.float32), 'bn_bias')
bn_mean = from_array(np.zeros(OUT_C, dtype=np.float32), 'bn_mean')
bn_var = from_array(np.ones(OUT_C, dtype=np.float32), 'bn_var')

X = make_tensor_value_info('X', TensorProto.FLOAT, [1, IN_C, 32, 32])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, None)

nodes = [
    make_node('Conv', ['X', 'conv_W', 'conv_B'], ['conv_out'],
             kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
    make_node('BatchNormalization',
             ['conv_out', 'bn_scale', 'bn_bias', 'bn_mean', 'bn_var'],
             ['bn_out'], epsilon=1e-5),
    make_node('Relu', ['bn_out'], ['relu_out']),
    make_node('MaxPool', ['relu_out'], ['Y'],
             kernel_shape=[2, 2], strides=[2, 2]),
]

inits = [conv_W, conv_B, bn_scale, bn_bias, bn_mean, bn_var]
g = make_graph(nodes, 'cnn_block', [X], [Y], initializer=inits)
model_cnn = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(model_cnn)

# Shape inference to see intermediate dims
model_cnn = shape_inference.infer_shapes(model_cnn)

print('Pipeline: Conv → BN → Relu → MaxPool')
print('\nNode Details:')
for i, node in enumerate(model_cnn.graph.node):
    attrs = {a.name: (list(a.ints) if a.ints else a.f if a.type == 1 else a.i)
             for a in node.attribute}
    print(f'  [{i}] {node.op_type:25s} '
          f'inputs={list(node.input)[:3]}... '
          f'out={list(node.output)}')
    if attrs:
        print(f'      attrs: {attrs}')

# Run
sess = ort.InferenceSession(
    model_cnn.SerializeToString(), providers=['CPUExecutionProvider'])
x = np.random.randn(1, IN_C, 32, 32).astype(np.float32)
y = sess.run(None, {'X': x})[0]
print(f'\nInput:  {x.shape}')
print(f'Output: {y.shape}')
assert y.shape == (1, OUT_C, 16, 16), 'MaxPool(2,2) halves spatial dims'
assert (y >= 0).all(), 'After Relu all values >= 0'

## Exercise 2: Data Flow Tracing

Track every named tensor through the graph and show its shape.

In [ ]:
def trace_tensors(model):
    """Map every named tensor to its shape and producing node."""
    shaped = shape_inference.infer_shapes(model)

    def dims(type_proto):
        if type_proto.tensor_type.HasField('shape'):
            return [d.dim_param or d.dim_value
                    for d in type_proto.tensor_type.shape.dim]
        return None

    shapes = {}
    for inp in shaped.graph.input:
        shapes[inp.name] = ('INPUT', dims(inp.type))
    for vi in shaped.graph.value_info:
        shapes[vi.name] = ('INTERMEDIATE', dims(vi.type))
    for out in shaped.graph.output:
        shapes[out.name] = ('OUTPUT', dims(out.type))

    # Map producer
    producers = {}
    for node in shaped.graph.node:
        for o in node.output:
            producers[o] = node.op_type

    print(f'{"Tensor":15s} {"Kind":15s} {"Producer":20s} {"Shape"}')
    print('-' * 70)
    init_names = {i.name for i in model.graph.initializer}
    for name in sorted(shapes.keys()):
        if name in init_names:
            continue
        kind, shape = shapes[name]
        prod = producers.get(name, 'input')
        print(f'  {name:15s} {kind:15s} {prod:20s} {shape}')

trace_tensors(model_cnn)

## Exercise 3: ONNX Tensor Type System

ONNX supports multiple data types. Let's explore them and test type compatibility.

In [ ]:
# All ONNX data types
type_map = {
    'FLOAT': TensorProto.FLOAT,
    'DOUBLE': TensorProto.DOUBLE,
    'FLOAT16': TensorProto.FLOAT16,
    'INT8': TensorProto.INT8,
    'INT16': TensorProto.INT16,
    'INT32': TensorProto.INT32,
    'INT64': TensorProto.INT64,
    'UINT8': TensorProto.UINT8,
    'BOOL': TensorProto.BOOL,
    'STRING': TensorProto.STRING,
    'COMPLEX64': TensorProto.COMPLEX64,
    'COMPLEX128': TensorProto.COMPLEX128,
    'BFLOAT16': TensorProto.BFLOAT16,
}

print(f'{"ONNX Type":15s} {"Enum":>5s} {"NumPy":>15s} {"Bytes":>6s}')
print('-' * 45)
for name, enum_val in type_map.items():
    try:
        np_dtype = tensor_dtype_to_np_dtype(enum_val)
        nbytes = np_dtype.itemsize
        print(f'  {name:15s} {enum_val:>5d} {str(np_dtype):>15s} {nbytes:>6d}')
    except Exception:
        print(f'  {name:15s} {enum_val:>5d} {"N/A":>15s}')

# Test that Relu works on different float types
print('\nRelu compatibility test:')
for dtype_name, dtype_enum in [('FLOAT', TensorProto.FLOAT),
                                ('DOUBLE', TensorProto.DOUBLE)]:
    X = make_tensor_value_info('X', dtype_enum, [None, 4])
    Y = make_tensor_value_info('Y', dtype_enum, [None, 4])
    g = make_graph([make_node('Relu', ['X'], ['Y'])], 't', [X], [Y])
    m = make_model(g, opset_imports=[make_opsetid('', 17)])
    check_model(m)
    np_dtype = tensor_dtype_to_np_dtype(dtype_enum)
    sess = ort.InferenceSession(
        m.SerializeToString(), providers=['CPUExecutionProvider'])
    x = np.array([[-1, 2, -3, 4]], dtype=np_dtype)
    r = sess.run(None, {'X': x})[0]
    print(f'  {dtype_name:8s}: input={x.dtype} output={r.dtype} correct={np.allclose(r, np.maximum(0, x))}')

## Exercise 4: NodeProto Inspection

Dissect the internal structure of `NodeProto` objects.

In [ ]:
def inspect_node(node, index=0):
    """Print detailed NodeProto information."""
    print(f'Node [{index}]:')
    print(f'  op_type:  {node.op_type}')
    print(f'  domain:   "{node.domain}"' if node.domain else '  domain:   (default)')
    print(f'  name:     "{node.name}"' if node.name else '  name:     (unnamed)')
    print(f'  inputs:   {list(node.input)}')
    print(f'  outputs:  {list(node.output)}')

    if node.attribute:
        print(f'  attributes ({len(node.attribute)}):')
        attr_type_names = {
            1: 'FLOAT', 2: 'INT', 3: 'STRING',
            4: 'TENSOR', 5: 'GRAPH', 6: 'SPARSE_TENSOR',
            7: 'FLOATS', 8: 'INTS', 9: 'STRINGS',
        }
        for attr in node.attribute:
            atype = attr_type_names.get(attr.type, f'TYPE_{attr.type}')
            if attr.type == 1:
                val = attr.f
            elif attr.type == 2:
                val = attr.i
            elif attr.type == 7:
                val = list(attr.floats)
            elif attr.type == 8:
                val = list(attr.ints)
            else:
                val = '...'
            print(f'    {attr.name:20s} ({atype:8s}) = {val}')
    print()

# Inspect all nodes in the CNN block
for i, node in enumerate(model_cnn.graph.node):
    inspect_node(node, i)

## Exercise 5: Edge Analysis

Build a complete producer/consumer map showing how tensors flow between nodes.

In [ ]:
def analyze_edges(graph):
    """Build complete edge analysis with fan-in/fan-out."""
    producers = {}  # tensor_name → (node_idx, op_type)
    consumers = {}  # tensor_name → [(node_idx, op_type)]

    init_names = {i.name for i in graph.initializer}

    for i, node in enumerate(graph.node):
        for out in node.output:
            producers[out] = (i, node.op_type)
        for inp in node.input:
            if inp not in consumers:
                consumers[inp] = []
            consumers[inp].append((i, node.op_type))

    # Find tensors with multiple consumers (fan-out)
    fan_out = {t: c for t, c in consumers.items() if len(c) > 1}

    print('Tensor Flow Analysis:')
    print('=' * 70)

    all_tensors = set()
    for node in graph.node:
        all_tensors.update(node.input)
        all_tensors.update(node.output)

    for tensor in sorted(all_tensors):
        if not tensor:
            continue
        src = producers.get(tensor, ('input/init', '-'))
        dsts = consumers.get(tensor, [])
        kind = 'init' if tensor in init_names else 'data'
        dst_str = ', '.join(f'{op}[{i}]' for i, op in dsts) if dsts else 'output'
        src_str = f'{src[1]}[{src[0]}]' if isinstance(src[0], int) else src[0]
        print(f'  {tensor:15s} ({kind:4s})  {src_str:20s} → {dst_str}')

    if fan_out:
        print(f'\nFan-out tensors (used by multiple nodes):')
        for t, cs in fan_out.items():
            print(f'  {t}: consumed by {len(cs)} nodes')

analyze_edges(model_cnn.graph)

## Exercise 6: Tensor Shape Manipulation

Use `Reshape`, `Squeeze`, `Unsqueeze`, and `Flatten` to transform tensor shapes.

In [ ]:
# Reshape: (2, 3, 4) → (2, 12)
X = make_tensor_value_info('X', TensorProto.FLOAT, [2, 3, 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [2, 12])
shape_init = from_array(np.array([2, 12], dtype=np.int64), 'shape')

g = make_graph(
    [make_node('Reshape', ['X', 'shape'], ['Y'])],
    'reshape', [X], [Y], [shape_init])
m = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(m)

sess = ort.InferenceSession(
    m.SerializeToString(), providers=['CPUExecutionProvider'])
x = np.random.randn(2, 3, 4).astype(np.float32)
r = sess.run(None, {'X': x})[0]
print(f'Reshape: {x.shape} → {r.shape}')
assert r.shape == (2, 12)
assert np.allclose(r, x.reshape(2, 12))

# Flatten: (2, 3, 4) → (2, 12) flattening from axis=1
X2 = make_tensor_value_info('X', TensorProto.FLOAT, [2, 3, 4])
Y2 = make_tensor_value_info('Y', TensorProto.FLOAT, [2, 12])
g2 = make_graph(
    [make_node('Flatten', ['X'], ['Y'], axis=1)],
    'flatten', [X2], [Y2])
m2 = make_model(g2, opset_imports=[make_opsetid('', 17)])
sess2 = ort.InferenceSession(
    m2.SerializeToString(), providers=['CPUExecutionProvider'])
r2 = sess2.run(None, {'X': x})[0]
print(f'Flatten: {x.shape} → {r2.shape}')
assert np.allclose(r2, x.reshape(2, -1))

# Unsqueeze: (4,) → (1, 4)
X3 = make_tensor_value_info('X', TensorProto.FLOAT, [4])
Y3 = make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
axes_init = from_array(np.array([0], dtype=np.int64), 'axes')
g3 = make_graph(
    [make_node('Unsqueeze', ['X', 'axes'], ['Y'])],
    'unsqueeze', [X3], [Y3], [axes_init])
m3 = make_model(g3, opset_imports=[make_opsetid('', 17)])
sess3 = ort.InferenceSession(
    m3.SerializeToString(), providers=['CPUExecutionProvider'])
x3 = np.array([1, 2, 3, 4], dtype=np.float32)
r3 = sess3.run(None, {'X': x3})[0]
print(f'Unsqueeze: {x3.shape} → {r3.shape}')
assert r3.shape == (1, 4)

print('\nAll shape manipulation ops verified!')

## Exercise 7: Multi-Output Nodes

Some ONNX operators produce multiple outputs (e.g., `Split`, `TopK`).

In [ ]:
# Split: divide a tensor into multiple parts
X = make_tensor_value_info('X', TensorProto.FLOAT, [6, 4])
Y1 = make_tensor_value_info('Y1', TensorProto.FLOAT, [2, 4])
Y2 = make_tensor_value_info('Y2', TensorProto.FLOAT, [2, 4])
Y3 = make_tensor_value_info('Y3', TensorProto.FLOAT, [2, 4])

split_init = from_array(np.array([2, 2, 2], dtype=np.int64), 'split')

g = make_graph(
    [make_node('Split', ['X', 'split'], ['Y1', 'Y2', 'Y3'], axis=0)],
    'multi_out', [X], [Y1, Y2, Y3], [split_init])
model_split = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(model_split)

sess_s = ort.InferenceSession(
    model_split.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.arange(24, dtype=np.float32).reshape(6, 4)
y1, y2, y3 = sess_s.run(None, {'X': x})

print(f'Input shape:  {x.shape}')
print(f'Output 1:     {y1.shape} = {y1.tolist()}')
print(f'Output 2:     {y2.shape} = {y2.tolist()}')
print(f'Output 3:     {y3.shape} = {y3.tolist()}')

assert np.allclose(y1, x[:2])
assert np.allclose(y2, x[2:4])
assert np.allclose(y3, x[4:6])

# TopK: returns values and indices
print('\n--- TopK (multi-output) ---')
X_tk = make_tensor_value_info('X', TensorProto.FLOAT, [3, 5])
V = make_tensor_value_info('Values', TensorProto.FLOAT, [3, 2])
I = make_tensor_value_info('Indices', TensorProto.INT64, [3, 2])
k_init = from_array(np.array([2], dtype=np.int64), 'K')

g_tk = make_graph(
    [make_node('TopK', ['X', 'K'], ['Values', 'Indices'], axis=-1)],
    'topk', [X_tk], [V, I], [k_init])
m_tk = make_model(g_tk, opset_imports=[make_opsetid('', 17)])

sess_tk = ort.InferenceSession(
    m_tk.SerializeToString(), providers=['CPUExecutionProvider'])
x_tk = np.array([[1, 5, 3, 9, 2],
                  [8, 4, 6, 1, 7],
                  [3, 2, 9, 5, 1]], dtype=np.float32)
vals, idxs = sess_tk.run(None, {'X': x_tk})
print(f'Input:\n{x_tk}')
print(f'Top-2 values:  {vals.tolist()}')
print(f'Top-2 indices: {idxs.tolist()}')

## Challenge: Full CNN Feature Extractor

Build a complete `Conv → BN → Relu → MaxPool → Flatten → Dense` pipeline.

```
X (1,3,32,32)
  │
  ▼ Conv(3→16, 3×3, pad=1)
(1,16,32,32)
  │
  ▼ BatchNorm
  │
  ▼ Relu
  │
  ▼ MaxPool(2×2)
(1,16,16,16)
  │
  ▼ Flatten(axis=1)
(1,4096)
  │
  ▼ MatMul + Add
(1,10)
```

In [ ]:
import time
np.random.seed(0)

OUT_C = 16
FLAT_DIM = OUT_C * 16 * 16
NUM_CLASSES = 10

# All initializers
all_inits = [
    from_array(np.random.randn(OUT_C, 3, 3, 3).astype(np.float32) * 0.1, 'cW'),
    from_array(np.zeros(OUT_C, dtype=np.float32), 'cB'),
    from_array(np.ones(OUT_C, dtype=np.float32), 'bn_s'),
    from_array(np.zeros(OUT_C, dtype=np.float32), 'bn_b'),
    from_array(np.zeros(OUT_C, dtype=np.float32), 'bn_m'),
    from_array(np.ones(OUT_C, dtype=np.float32), 'bn_v'),
    from_array(np.random.randn(FLAT_DIM, NUM_CLASSES).astype(np.float32) * 0.01, 'fc_W'),
    from_array(np.zeros(NUM_CLASSES, dtype=np.float32), 'fc_b'),
]

X = make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 32, 32])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1, NUM_CLASSES])

full_nodes = [
    make_node('Conv', ['X', 'cW', 'cB'], ['conv_out'],
             kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
    make_node('BatchNormalization',
             ['conv_out', 'bn_s', 'bn_b', 'bn_m', 'bn_v'],
             ['bn_out'], epsilon=1e-5),
    make_node('Relu', ['bn_out'], ['relu_out']),
    make_node('MaxPool', ['relu_out'], ['pool_out'],
             kernel_shape=[2, 2], strides=[2, 2]),
    make_node('Flatten', ['pool_out'], ['flat'], axis=1),
    make_node('MatMul', ['flat', 'fc_W'], ['logits_pre']),
    make_node('Add', ['logits_pre', 'fc_b'], ['Y']),
]

g_full = make_graph(full_nodes, 'cnn_classifier', [X], [Y], all_inits)
model_full = make_model(g_full, opset_imports=[make_opsetid('', 17)])
check_model(model_full)
model_full = shape_inference.infer_shapes(model_full)

# Trace data flow
print('=== CNN Feature Extractor ===')
trace_tensors(model_full)

# Run inference
sess_full = ort.InferenceSession(
    model_full.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.random.randn(1, 3, 32, 32).astype(np.float32)
y = sess_full.run(None, {'X': x})[0]
print(f'\nInput:  {x.shape}')
print(f'Output: {y.shape} (logits for {NUM_CLASSES} classes)')

# Benchmark
sess_full.run(None, {'X': x})  # warmup
times = []
for _ in range(300):
    t0 = time.perf_counter()
    sess_full.run(None, {'X': x})
    times.append((time.perf_counter() - t0) * 1e6)

print(f'\nBenchmark (batch=1):')
print(f'  Mean:   {np.mean(times):.1f} us')
print(f'  P50:    {np.percentile(times, 50):.1f} us')
print(f'  P99:    {np.percentile(times, 99):.1f} us')

total_params = sum(to_array(i).size for i in model_full.graph.initializer)
print(f'  Params: {total_params:,}')

---

## Summary

| Exercise | Topic | Key Takeaway |
|----------|-------|-------------|
| 1 | CNN block | Conv → BN → Relu → MaxPool |
| 2 | Tensor tracing | Map names to shapes and producers |
| 3 | Type system | FLOAT, DOUBLE, INT64, BOOL, etc. |
| 4 | NodeProto | op_type, inputs, outputs, attributes |
| 5 | Edge analysis | Producer/consumer relationships |
| 6 | Shape ops | Reshape, Flatten, Squeeze, Unsqueeze |
| 7 | Multi-output | Split, TopK produce multiple tensors |
| Challenge | Full CNN | Complete feature extractor pipeline |

**Next:** [ONNX IR Specification](../03_ONNX_IR_Specification/)